# Data cleaning & revenue normalization — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## 1. Import e percorsi

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "deep_dives":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name.lower() in {"cleaning", "eda", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXCEL_NAME = "Adidas US Sales Datasets.xlsx"
source_candidates = [PROJECT_ROOT / "data" / EXCEL_NAME, PROJECT_ROOT / EXCEL_NAME]
SOURCE_FILE = next((p for p in source_candidates if p.is_file()), None)
if SOURCE_FILE is None:
    raise FileNotFoundError(f"File non trovato. Verifica: {source_candidates}")

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_FILE = OUTPUT_DIR / "adidas_sales_clean.csv"
print("Input:", SOURCE_FILE)
print("Output:", CLEAN_FILE)

## 2. Caricamento e verifica delle colonne

In [ ]:
raw = pd.read_excel(
    SOURCE_FILE,
    sheet_name="ADIdata_RAW",
    header=4,
    usecols="B:N",
    dtype={"Retailer ID": "string"},
    engine="openpyxl",
)

raw.columns = (
    raw.columns.astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

EXPECTED_COLS = [
    "retailer", "retailer_id", "invoice_date", "region", "state", "city",
    "product", "price_per_unit", "units_sold", "total_sales",
    "operating_profit", "operating_margin", "sales_method",
]
missing_cols = set(EXPECTED_COLS) - set(raw.columns)
if missing_cols:
    raise ValueError(f"Colonne attese assenti: {sorted(missing_cols)}")

df = raw[EXPECTED_COLS].copy()
initial_rows = len(df)
print("Dimensioni iniziali:", df.shape)
print("Valori nulli per colonna (prima della pulizia):")
print(df.isna().sum().to_string())
display(df.head(3))

## 3. Rimozione delle righe vuote e normalizzazione dei testi

In [ ]:
df = df.dropna(how="all").copy()

TEXT_COLS = ["retailer", "retailer_id", "region", "state", "city", "product", "sales_method"]
NULL_TOKENS = {"", "none", "null", "nan", "n/a", "na", "-"}

for col in TEXT_COLS:
    series = (
        df[col].astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    df[col] = series.mask(series.str.casefold().isin(NULL_TOKENS))

df["retailer_id"] = df["retailer_id"].str.replace(r"\.0$", "", regex=True)

CANONICAL_LABELS = {
    "retailer": ["Foot Locker", "West Gear", "Sports Direct", "Kohl's", "Amazon", "Walmart"],
    "region": ["West", "Northeast", "Midwest", "South", "Southeast"],
    "product": [
        "Men's Street Footwear", "Men's Athletic Footwear", "Men's Apparel",
        "Women's Street Footwear", "Women's Athletic Footwear", "Women's Apparel",
    ],
    "sales_method": ["Online", "Outlet", "In-store"],
}

for col, labels in CANONICAL_LABELS.items():
    canonical_map = {label.casefold(): label for label in labels}
    df[col] = df[col].map(
        lambda value: canonical_map.get(value.casefold(), value) if pd.notna(value) else pd.NA
    ).astype("string")

for col in ["state", "city"]:
    df[col] = df[col].str.title()

print("Testi uniformati; nessuna categoria sconosciuta viene cancellata automaticamente.")

## 4. Conversione dei tipi di dato e delle date

In [ ]:
FINANCIAL_COLS = ["price_per_unit", "units_sold", "total_sales", "operating_profit", "operating_margin"]

for col in FINANCIAL_COLS:
    cleaned = (
        df[col].astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(cleaned, errors="coerce")

raw_date = df["invoice_date"]
numeric_date = pd.to_numeric(raw_date, errors="coerce")
is_excel_serial = numeric_date.between(20_000, 80_000)
parsed_date = pd.to_datetime(raw_date, errors="coerce")
parsed_date.loc[is_excel_serial] = pd.to_datetime(
    numeric_date.loc[is_excel_serial], unit="D", origin="1899-12-30", errors="coerce"
)
df["invoice_date"] = parsed_date.dt.normalize()

for col in ["price_per_unit", "total_sales", "operating_profit"]:
    df[col] = df[col].round(2)
df["operating_margin"] = df["operating_margin"].round(4)

print(df.dtypes.to_string())

## 5. Gestione di valori nulli e valori non validi

In [ ]:
MANDATORY_COLS = [
    "retailer", "retailer_id", "invoice_date", "region", "state", "city",
    "product", "price_per_unit", "units_sold", "sales_method",
]

invalid_missing = df[MANDATORY_COLS].isna().any(axis=1)
invalid_price = df["price_per_unit"].lt(0).fillna(False)
invalid_units = (
    df["units_sold"].lt(0).fillna(False)
    | df["units_sold"].mod(1).ne(0).fillna(False)
)
invalid_mask = invalid_missing | invalid_price | invalid_units
rejected = df.loc[invalid_mask].copy()

if not rejected.empty:
    rejected.to_csv(OUTPUT_DIR / "adidas_rejected_rows.csv", index=False, date_format="%Y-%m-%d")

df = df.loc[~invalid_mask].copy()
df["units_sold"] = df["units_sold"].astype("int64")

print("Record scartati per dati obbligatori assenti o non validi:", len(rejected))
print("Record validi rimasti:", len(df))
print("Unità vendute uguali a zero (mantenute):", int(df["units_sold"].eq(0).sum()))

## 6. Rimozione dei duplicati

In [ ]:
duplicate_mask = df.duplicated(keep="first")
num_duplicates = int(duplicate_mask.sum())
df = df.loc[~duplicate_mask].reset_index(drop=True)
print("Duplicati completi rimossi:", num_duplicates)
print("Righe dopo deduplicazione:", len(df))

## 7. Ricalcolo di `total_sales`

In [ ]:
old_total_sales = df["total_sales"].copy()
recalculated_total_sales = (df["price_per_unit"] * df["units_sold"]).round(2)

diff = (old_total_sales - recalculated_total_sales).abs()
mismatch_mask = (old_total_sales.notna() & diff.gt(0.01)).fillna(False)
num_mismatch = int(mismatch_mask.sum())

tenfold_mask = (
    recalculated_total_sales.gt(0)
    & (old_total_sales / recalculated_total_sales).sub(10).abs().lt(0.000001)
).fillna(False)

print("Ricavi originali differenti dalla formula corretta:", num_mismatch)
print("Di cui con errore di fattore 10:", int(tenfold_mask.sum()))

old_position = df.columns.get_loc("total_sales")
df = df.drop(columns="total_sales")
df.insert(old_position, "total_sales", recalculated_total_sales)

display(df[["invoice_date", "product", "price_per_unit", "units_sold", "total_sales"]].head(5))

## 8. Ricalcolo di `operating_profit`

In [ ]:
if df["operating_margin"].isna().any() or not df["operating_margin"].between(0, 1).all():
    raise ValueError("Operating Margin mancante o fuori dall'intervallo [0, 1]")

old_operating_profit = df["operating_profit"].copy()
recalculated_operating_profit = (df["total_sales"] * df["operating_margin"]).round(2)

num_profit_changed = int(
    (old_operating_profit.isna() | (old_operating_profit - recalculated_operating_profit).abs().gt(0.01)).sum()
)

profit_position = df.columns.get_loc("operating_profit")
df = df.drop(columns="operating_profit")
df.insert(profit_position, "operating_profit", recalculated_operating_profit)

print("Operating Profit ricalcolati:", len(df))
print("Valori diversi dagli originali:", num_profit_changed)
display(df[["total_sales", "operating_margin", "operating_profit"]].head(5))

## 9. Controlli di coerenza

In [ ]:
margin_diff = (
    df["operating_profit"] - (df["total_sales"] * df["operating_margin"]).round(2)
).abs()

print("Righe con ricavi non coerenti:", int(
    (df["total_sales"] - df["price_per_unit"] * df["units_sold"]).abs().gt(0.01).sum()
))
print("Righe con profitto e margine non coerenti:", int(margin_diff.gt(0.01).sum()))
print("Righe con profitto maggiore dei ricavi:", int(df["operating_profit"].gt(df["total_sales"]).sum()))

## 10. Validazione finale ed esportazione

In [ ]:
assert list(df.columns) == EXPECTED_COLS, "Schema delle colonne modificato involontariamente"
assert not df[MANDATORY_COLS].isna().any().any(), "Restano valori nulli obbligatori"
assert not df.duplicated().any(), "Sono presenti duplicati integrali"
assert pd.api.types.is_datetime64_any_dtype(df["invoice_date"]), "Date non convertite"
assert df["price_per_unit"].ge(0).all(), "Prezzi negativi"
assert df["units_sold"].ge(0).all(), "Quantità negative"
assert np.allclose(
    df["total_sales"].to_numpy(dtype=float),
    (df["price_per_unit"] * df["units_sold"]).round(2).to_numpy(dtype=float),
    rtol=0, atol=0.001,
), "Ricalcolo ricavi non coerente"
assert np.allclose(
    df["operating_profit"].to_numpy(dtype=float),
    (df["total_sales"] * df["operating_margin"]).round(2).to_numpy(dtype=float),
    rtol=0, atol=0.001,
), "Ricalcolo profitto operativo non coerente"
assert len(df) + len(rejected) + num_duplicates <= initial_rows

export_df = df.copy()
export_df["invoice_date"] = export_df["invoice_date"].dt.strftime("%Y-%m-%d")
assert export_df["invoice_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
export_df.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")

print("CSV salvato:", CLEAN_FILE)
print("Righe:", len(export_df), "| Colonne:", export_df.shape[1])
display(export_df.head(5))